<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LotteryTicketSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==== SETUP ====
from google.colab import drive
drive.mount('/content/drive')

!pip install --upgrade transformers accelerate tqdm

import torch
import copy
import json
import re
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn.utils import prune
import torch.nn.functional as F

# === PARAMETER & PFADEN ===
MODEL_PATH    = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
INPUT_JSON    = "/content/drive/MyDrive/Colab Notebooks/12B_combined_golden.json"
PRUNED_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH"
OUTPUT_JSON   = "/content/drive/MyDrive/Colab Notebooks/lth-qwen3-4b_testresults.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_ITER = 3          # Anzahl der LTH-Pruning-Iterationen
PRUNE_RATE = 0.2      # Prune-Rate pro Iteration
EPOCHS_PER_ITER = 1   # Anzahl Epochen pro Iteration (für Demo niedrig, bei echten Experimenten 2–3+)
BATCH_SIZE = 4        # Training batch size

# ==== 1. Modell & Tokenizer laden, Initialisierung speichern ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map=DEVICE
)
model.train()
init_state = copy.deepcopy(model.state_dict())  # LTH: Merke Initialisierung!

In [ ]:
# ==== 2. Hilfsfunktionen: Datensatz-Vorbereitung ====
def get_training_examples(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        questions = json.load(f)['questions']
    # Wir nehmen jeweils nur q['body'] und q['snippets'], für Demo (Anpassung empfohlen)
    texts = []
    for q in questions:
        qtxt = q['body']
        ctx = "\n".join([s['text'] for s in q.get('snippets', [])[:2]])
        target = q.get('exact_answer', None)
        # Compose simple QA style prompt (kannst du optimieren!)
        if ctx:
            prompt = f"Question: {qtxt}\nContext:\n{ctx}\nAnswer:"
        else:
            prompt = f"Question: {qtxt}\nAnswer:"
        if isinstance(target, list):
            target = ", ".join([str(x) for x in target])
        elif target is None:
            target = ""
        texts.append( (prompt, str(target)) )
    return texts

train_examples = get_training_examples(INPUT_JSON)

# ==== 3. Trainings-/Finetuning-Funktion ====
def make_supervised_data(examples, tokenizer, max_length=512, max_target_length=64):
    input_ids_list, labels_list = [], []
    for prompt, target in examples:
        full_text = prompt + " " + target
        tokenized = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length + max_target_length,
            padding="max_length"
        )
        input_ids = tokenized['input_ids'][0]
        prompt_ids = tokenizer(prompt, truncation=True, max_length=max_length, add_special_tokens=False)['input_ids']
        label = input_ids.clone()
        label[:len(prompt_ids)] = -100
        input_ids_list.append(input_ids)
        labels_list.append(label)
    input_ids = torch.stack(input_ids_list)
    labels = torch.stack(labels_list)
    return input_ids, labels

def finetune(model, tokenizer, train_data, num_epochs=1, batch_size=4, lr=2e-5, max_length=512, max_target_length=64):
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = (len(train_data) // batch_size) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    losses = []
    for epoch in range(num_epochs):
        pbar = tqdm(range(0, len(train_data), batch_size), desc=f'Epoch {epoch+1}/{num_epochs}')
        for i in pbar:
            batch = train_data[i:i+batch_size]
            input_ids, labels = make_supervised_data(batch, tokenizer, max_length, max_target_length)
            input_ids = input_ids.to(model.device)
            labels = labels.to(model.device)
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix(loss=loss.item())
            losses.append(loss.item())
    return losses

# ==== 4. LTH: Iterativer Pruning-Reset-Train-Loop ====
def apply_magnitude_pruning(model, amount=0.2):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")
    return model

for it in range(NUM_ITER):
    print(f"\n=== LTH Iteration {it+1}/{NUM_ITER} ===")
    # a) Trainiere Modell (mit aktuellem Maskensubnetz)
    finetune(model, tokenizer, train_examples, num_epochs=EPOCHS_PER_ITER, batch_size=BATCH_SIZE)
    # b) Prune p% (Magnitude)
    model = apply_magnitude_pruning(model, amount=PRUNE_RATE)
    print(f"Pruning durchgeführt: {PRUNE_RATE*100:.1f}% der Gewichte entfernt.")
    # c) Setze überlebende Gewichte zurück auf Initialisierung (Maske bleibt)
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' in name and param.dim() > 1:
                mask = (param != 0)
                param.data[mask] = init_state[name][mask].data.clone()
    print("Maskengewichte auf Initialisierung zurückgesetzt.")

# ==== 5. Finale Phase: „Winning Ticket“ feintunen ====
print("\n==== Feintuning des finalen Winning Tickets ====")
finetune(model, tokenizer, train_examples, num_epochs=EPOCHS_PER_ITER, batch_size=BATCH_SIZE)

# ==== 6. Gepruntes Modell speichern ====
model.save_pretrained(PRUNED_MODEL_PATH)
print(f"Gepruntes LTH-Modell gespeichert unter {PRUNED_MODEL_PATH}")

# ==== 7. Evaluation (wie gehabt) ====
gen_conf = GenerationConfig(
    max_new_tokens=200,
    do_sample=False
)

def build_messages(qtext, snippets, qtype, mode="exact"):
    system_msg = {"role":"system","content":"/no_think"}
    ctx = "\n".join(s["text"] for s in snippets[:2])
    if mode == "exact":
        if qtype == "yesno":
            content = f"Question: {qtext}\nContext:\n{ctx}\nAnswer only 'yes' or 'no', in English, no extras."
        elif qtype == "factoid":
            content = f"Question: {qtext}\nContext:\n{ctx}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
        elif qtype == "list":
            content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a comma-separated list of relevant items, in English, no filler words."
        else:
            content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a brief answer in English."
    else:  # ideal
        if qtype == "yesno":
            content = f"Question: {qtext}\nContext:\n{ctx}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
        else:
            content = f"Question: {qtext}\nContext:\n{ctx}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."
    user_msg = {"role":"user","content":content}
    return [system_msg, user_msg]

def clean_exact(text, qtype):
    txt = text.strip()
    txt = re.sub(r'<\/think>','', txt)
    txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)
    if qtype == "yesno":
        return "yes" if txt.lower().startswith("yes") else "no"
    if qtype in ("factoid","list"):
        items = [i.strip() for i in txt.split(",") if i.strip()]
        return items
    return None

def clean_ideal(text, qtype):
    txt = text.strip()
    txt = re.sub(r'<\/think>','', txt)
    txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)
    sentences = re.split(r'(?<=[.!?])\s+', txt)
    if qtype == "yesno":
        return sentences[0].strip()
    total = 0
    out = []
    for sent in sentences:
        length = len(sent.split())
        if total + length <= 200:
            out.append(sent)
            total += length
        else:
            break
    return " ".join(out).strip()

with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    questions = json.load(f)['questions']

submission = []

for i in tqdm(range(0, len(questions), BATCH_SIZE), desc='Batches'):
    batch = questions[i:i+BATCH_SIZE]
    msgs_ex = [build_messages(q['body'], q.get('snippets',[]), q['type'], mode='exact') for q in batch]
    texts_ex = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in msgs_ex]
    inputs_ex = tokenizer(texts_ex, return_tensors='pt', padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        out_ex = model.generate(**inputs_ex, generation_config=gen_conf)
    dec_ex = []
    for idx, q in enumerate(batch):
        start = inputs_ex['input_ids'].shape[1]
        ids = out_ex[idx][start:].tolist()
        text = tokenizer.decode(ids, skip_special_tokens=True)
        dec_ex.append(clean_exact(text, q['type']))
    # ideal
    msgs_id = [build_messages(q['body'], q.get('snippets',[]), q['type'], mode='ideal') for q in batch]
    texts_id = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in msgs_id]
    inputs_id = tokenizer(texts_id, return_tensors='pt', padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        out_id = model.generate(**inputs_id, generation_config=gen_conf)
    dec_id = []
    for idx, q in enumerate(batch):
        start = inputs_id['input_ids'].shape[1]
        ids = out_id[idx][start:].tolist()
        text = tokenizer.decode(ids, skip_special_tokens=True)
        dec_id.append(clean_ideal(text, q['type']))
    for idx, q in enumerate(batch):
        submission.append({
            'id': q['id'],
            'type': q['type'],
            'exact_answer': q.get('exact_answer'),
            'ideal_answer': q.get('ideal_answer'),
            'exact_prediction': dec_ex[idx],
            'ideal_prediction': dec_id[idx]
        })

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
print('✅ Submission file created at:', OUTPUT_JSON)

# ==== 8. Modellstatistiken ====
def compute_pruned_stats(model):
    total_params = 0
    nonzero_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() > 1 and "weight" in name:
            total_params += param.numel()
            nonzero_params += torch.count_nonzero(param).item()
    zero_params = total_params - nonzero_params
    sparsity = 100.0 * zero_params / total_params
    compression_ratio = total_params / nonzero_params if nonzero_params > 0 else float("inf")
    print(f"Gesamtparameter: {total_params:,}")
    print(f"Aktive Parameter (<> 0): {nonzero_params:,}")
    print(f"Sparsity: {sparsity:.2f}%")
    print(f"Komprimierungsrate: {compression_ratio:.2f}x")
    return total_params, nonzero_params, sparsity, compression_ratio

compute_pruned_stats(model)
